# 复习调度测试台

这个 notebook 直接连接之前整理好的例题和复习 seed，测试两种复习模式：

- 错题模式 `question_first`
- 知识点模式 `leaf_first`

说明：
- 这里会先复制一份 seed，再随机生成一批复习时间、掌握度、稳定度
- 默认只往 `scratch/` 写测试文件
- 不会改正式例题文件和正式 schema


In [1]:
import copy
import json
import random
import re
import sys
from datetime import timedelta
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from review_scheduler import build_review_schedule, now_from_value
from review_state_manager import apply_review_action

EXAMPLE_MD_PATH = PROJECT_ROOT / 'docs' / 'rag_samples' / 'taizhou_simulated_exam_examples_batch_01.md'
SEED_PATH = PROJECT_ROOT / 'docs' / 'rag_samples' / 'taizhou_simulated_exam_review_seed_batch_01.json'
SCRATCH_DIR = PROJECT_ROOT / 'scratch' / 'review_scheduler_playground'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

RANDOMIZED_STATE_PATH = SCRATCH_DIR / 'review_state_randomized.json'
QUESTION_MODE_OUTPUT_PATH = SCRATCH_DIR / 'question_first_schedule.json'
NODE_MODE_OUTPUT_PATH = SCRATCH_DIR / 'leaf_first_schedule.json'


## 1. 载入例题和复习 seed

这里把之前的例题 markdown 解析成 `question_id -> 题目/答案/解析`，后面展示调度结果时会直接引用。

In [2]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def split_question_sections(text: str) -> list[tuple[str, str]]:
    pattern = re.compile(r'^##\s+(q\d+)\s*$', re.MULTILINE)
    matches = list(pattern.finditer(text))
    sections = []
    for index, match in enumerate(matches):
        question_id = match.group(1)
        start = match.end()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        sections.append((question_id, text[start:end].strip()))
    return sections


def extract_field(block: str, label: str) -> str:
    pattern = re.compile(rf'- {re.escape(label)}：\n(.*?)(?=\n- [^\n]+：|\Z)', re.DOTALL)
    match = pattern.search(block)
    if not match:
        return ''
    return match.group(1).strip()


def load_example_map(path: Path) -> dict[str, dict[str, str]]:
    text = path.read_text(encoding='utf-8')
    mapping = {}
    for question_id, block in split_question_sections(text):
        mapping[question_id] = {
            'stem': extract_field(block, '题目'),
            'question_type': extract_field(block, '题型'),
            'correct_answer': extract_field(block, '标准答案'),
            'solution_text': extract_field(block, '参考解析'),
        }
    return mapping


BASE_STATE = load_json(SEED_PATH)
EXAMPLE_MAP = load_example_map(EXAMPLE_MD_PATH)

print('已载入例题数量：', len(EXAMPLE_MAP))
print('题目 ID：', sorted(EXAMPLE_MAP.keys()))


已载入例题数量： 8
题目 ID： ['q001', 'q002', 'q003', 'q004', 'q005', 'q006', 'q007', 'q010']


## 2. 给例题和知识点随机标注复习时间

这里固定随机种子，生成一份可复现实验数据：

- `last_reviewed_at`
- `next_review_at`
- `mastery`
- `stability`
- `state`
- `last_result`

这样调度器就不会所有对象都长得一样。

In [3]:
def state_profile_for_node(rng: random.Random) -> tuple[str, float, float]:
    state = rng.choice(['learning', 'review', 'stable'])
    if state == 'learning':
        mastery = round(rng.uniform(0.15, 0.45), 2)
        stability = round(rng.uniform(0.4, 1.8), 2)
    elif state == 'review':
        mastery = round(rng.uniform(0.45, 0.75), 2)
        stability = round(rng.uniform(1.5, 4.5), 2)
    else:
        mastery = round(rng.uniform(0.75, 0.95), 2)
        stability = round(rng.uniform(4.0, 9.0), 2)
    return state, mastery, stability


def state_profile_for_question(rng: random.Random) -> tuple[str, str, float, float, int]:
    state = rng.choice(['learning', 'review', 'stable'])
    last_result = rng.choice(['correct', 'wrong', 'partial'])
    if state == 'learning':
        mastery = round(rng.uniform(0.10, 0.40), 2)
        stability = round(rng.uniform(0.3, 1.5), 2)
    elif state == 'review':
        mastery = round(rng.uniform(0.40, 0.70), 2)
        stability = round(rng.uniform(1.2, 3.5), 2)
    else:
        mastery = round(rng.uniform(0.70, 0.92), 2)
        stability = round(rng.uniform(3.0, 6.5), 2)
    review_count = rng.randint(1, 6)
    return state, last_result, mastery, stability, review_count


def randomize_review_state(seed_payload: dict, *, seed: int = 20260621, now_iso: str = '2026-06-21T10:00:00+08:00') -> tuple[dict, object]:
    rng = random.Random(seed)
    now = now_from_value(now_iso)
    payload = copy.deepcopy(seed_payload)

    for node_state in payload['knowledge_point_states']:
        state, mastery, stability = state_profile_for_node(rng)
        last_days_ago = rng.uniform(0.3, 15.0)
        last_reviewed_at = now - timedelta(days=last_days_ago)
        next_review_at = last_reviewed_at + timedelta(days=stability)
        node_state['state'] = state
        node_state['mastery'] = mastery
        node_state['stability'] = stability
        node_state['correct_count'] = rng.randint(0, 6)
        node_state['wrong_count'] = rng.randint(0, 4)
        node_state['last_reviewed_at'] = last_reviewed_at.isoformat()
        node_state['next_review_at'] = next_review_at.isoformat()

    for question_state in payload['example_question_states']:
        state, last_result, mastery, stability, review_count = state_profile_for_question(rng)
        last_days_ago = rng.uniform(0.2, 12.0)
        last_reviewed_at = now - timedelta(days=last_days_ago)
        next_review_at = last_reviewed_at + timedelta(days=stability)
        question_state['state'] = state
        question_state['last_result'] = last_result
        question_state['mastery'] = mastery
        question_state['stability'] = stability
        question_state['review_count'] = review_count
        question_state['last_reviewed_at'] = last_reviewed_at.isoformat()
        question_state['next_review_at'] = next_review_at.isoformat()

    payload['generated_at'] = now.isoformat()
    return payload, now


RANDOMIZED_STATE, DEMO_NOW = randomize_review_state(BASE_STATE)
RANDOMIZED_STATE_PATH.write_text(
    json.dumps(RANDOMIZED_STATE, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

print('随机化后的测试状态已写入：', RANDOMIZED_STATE_PATH)
print('调度参考时间：', DEMO_NOW.isoformat())
print('示例题目 q010 当前状态：')
for item in RANDOMIZED_STATE['example_question_states']:
    if item['question_id'] == 'q010':
        print(json.dumps(item, ensure_ascii=False, indent=2))
        break


随机化后的测试状态已写入： /Users/xumuchi/Desktop/TeachAgent/scratch/review_scheduler_playground/review_state_randomized.json
调度参考时间： 2026-06-21T10:00:00+08:00
示例题目 q010 当前状态：
{
  "question_id": "q010",
  "state": "learning",
  "linked_node_ids": [
    "math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化",
    "math.数列与不等式.数列.基础概念.数列递推公式解读"
  ],
  "primary_node_ids": [
    "math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化"
  ],
  "secondary_node_ids": [
    "math.数列与不等式.数列.基础概念.数列递推公式解读"
  ],
  "source_batch_id": "taizhou_simulated_exam_examples_batch_01",
  "last_result": "partial",
  "review_count": 5,
  "priority_note": "递推数列转化例题",
  "mastery": 0.27,
  "stability": 1.41,
  "last_reviewed_at": "2026-06-17T21:47:28.499944+08:00",
  "next_review_at": "2026-06-19T07:37:52.499944+08:00"
}


## 3. 错题模式 `question_first`

这个模式下，系统优先给题目。下面会展示：

- 题目优先级
- 主知识点
- 题目预览


In [5]:
QUESTION_PLAN = build_review_schedule(
    RANDOMIZED_STATE,
    now=DEMO_NOW,
    user_mode='question_first',
    question_limit=5,
    node_limit=5,
    bundle_limit=5,
    bundle_question_limit=2,
)

QUESTION_MODE_OUTPUT_PATH.write_text(
    json.dumps(QUESTION_PLAN.as_dict(), ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

print('错题模式输出已写入：', QUESTION_MODE_OUTPUT_PATH)
print()
print('一、推荐错题')
for rank, item in enumerate(QUESTION_PLAN.question_priorities, start=1):
    qid = item['question_id']
    example = EXAMPLE_MAP.get(qid, {})
    stem = example.get('stem', '').replace('\n', ' ')
    primary_node_ids = item.get('primary_node_ids', [])
    print(f'[{rank}] {qid} | 总分={item["priority_score"]} | 最近结果={item["last_result"]}')
    print('  主知识点：', primary_node_ids[0] if primary_node_ids else '无')
    print('  题目预览：', stem[:90] + ('...' if len(stem) > 90 else ''))
    print()


错题模式输出已写入： /Users/xumuchi/Desktop/TeachAgent/scratch/review_scheduler_playground/question_first_schedule.json

一、推荐错题
[1] q005 | 总分=0.8979 | 最近结果=wrong
  主知识点： math.计数_二项式_推理证明_逻辑与复数.简易逻辑与算法.充分必要条件.必要非充分条件
  题目预览： “\(a_1+a_5=2a_3\)”是“数列 \(\{a_n\}\) 为等差数列”的

[2] q001 | 总分=0.8554 | 最近结果=wrong
  主知识点： math.集合_映射_函数与导数.集合.集合交集运算
  题目预览： 设集合 \(A=\{x\mid |x|\ge 1\},\ B=\{x\mid x<2\}\)，则 \(A\cap B=\)

[3] q003 | 总分=0.8527 | 最近结果=wrong
  主知识点： math.空间几何.空间几何体.柱体.棱柱.正棱柱
  题目预览： 正六棱柱的底面边长为 \(6\)，高为 \(4\)。若挖去一个以上底面中心为顶点、下底面为底面的正六棱锥，则剩余几何体的体积为

[4] q010 | 总分=0.7788 | 最近结果=partial
  主知识点： math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化
  题目预览： 已知正项数列 \(\{a_n\}\) 满足   \[   a_{n+1}=\frac{a_n+\sqrt{a_n^2+1}}{2},   \]   则下列结论正确的是   A. \...

[5] q006 | 总分=0.7056 | 最近结果=correct
  主知识点： math.三角函数_平面向量与解三角形.平面向量.向量夹角公式
  题目预览： 已知向量 \(\vec a,\vec b,\vec c\) 均为单位向量，若 \(\vec a+\vec b+\vec c=0\)，则 \(\vec a\) 与 \(\vec b\...



## 4. 知识点模式 `leaf_first`

这个模式下，系统优先给知识点，并且必须同时带绑定题。

下面会展示：

- 知识点优先级
- 每个知识点对应的推荐题目 bundle


In [6]:
LEAF_PLAN = build_review_schedule(
    RANDOMIZED_STATE,
    now=DEMO_NOW,
    user_mode='leaf_first',
    question_limit=5,
    node_limit=5,
    bundle_limit=5,
    bundle_question_limit=3,
)

NODE_MODE_OUTPUT_PATH.write_text(
    json.dumps(LEAF_PLAN.as_dict(), ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

print('知识点模式输出已写入：', NODE_MODE_OUTPUT_PATH)
print()
print('一、推荐知识点')
for rank, node_item in enumerate(LEAF_PLAN.node_priorities, start=1):
    print(f'[{rank}] {node_item["node_id"]} | 总分={node_item["priority_score"]} | mastery={node_item["mastery"]}')
print()
print('二、知识点绑定题目 bundle')
for rank, bundle in enumerate(LEAF_PLAN.review_plan['recommended_bundles'], start=1):
    print(f'[{rank}] node_id = {bundle["node_id"]} | priority = {bundle["priority_score"]}')
    for question_id in bundle['question_ids']:
        example = EXAMPLE_MAP.get(question_id, {})
        stem = example.get('stem', '').replace('\n', ' ')
        print('  -', question_id, '|', stem[:80] + ('...' if len(stem) > 80 else ''))
    print()


知识点模式输出已写入： /Users/xumuchi/Desktop/TeachAgent/scratch/review_scheduler_playground/leaf_first_schedule.json

一、推荐知识点
[1] math.解析几何.圆的方程.圆的标准方程 | 总分=0.8659 | mastery=0.3
[2] math.集合_映射_函数与导数.集合.数轴表示集合关系 | 总分=0.84 | mastery=0.44
[3] math.解析几何.圆的方程.直线与圆的位置关系 | 总分=0.7697 | mastery=0.7
[4] math.三角函数_平面向量与解三角形.平面向量.向量夹角公式 | 总分=0.7117 | mastery=0.34
[5] math.空间几何.空间几何体.柱体.棱柱.正棱柱 | 总分=0.7001 | mastery=0.87

二、知识点绑定题目 bundle
[1] node_id = math.解析几何.圆的方程.圆的标准方程 | priority = 0.8659
  - q007 | 若圆 \((x+1)^2+(y-2)^2=1\) 上存在两个不同的点 \(M,N\)，直线 \(4x+3y+t=0\) 上存在一点 \(P\)，使得 \(\an...

[2] node_id = math.集合_映射_函数与导数.集合.数轴表示集合关系 | priority = 0.84
  - q001 | 设集合 \(A=\{x\mid |x|\ge 1\},\ B=\{x\mid x<2\}\)，则 \(A\cap B=\)

[3] node_id = math.解析几何.圆的方程.直线与圆的位置关系 | priority = 0.7697
  - q007 | 若圆 \((x+1)^2+(y-2)^2=1\) 上存在两个不同的点 \(M,N\)，直线 \(4x+3y+t=0\) 上存在一点 \(P\)，使得 \(\an...

[4] node_id = math.三角函数_平面向量与解三角形.平面向量.向量夹角公式 | priority = 0.7117
  - q006 | 已知向量 \(\vec a,\vec b,\vec c\) 均为单位向量，若 \(\vec a+\vec b+\vec c=

## 5. 手动按钮模拟（可选）

这里演示你刚才说的两种人工权力：

- `mark_important`
- `skip_temporarily`

这一步不是必须，但可以验证 UI 按钮以后怎么接。

In [7]:
important_state = copy.deepcopy(RANDOMIZED_STATE)
important_event = apply_review_action(
    important_state,
    action='mark_important',
    target_type='wrong_question',
    target_id='q010',
    now=DEMO_NOW,
)
important_plan = build_review_schedule(
    important_event.updated_payload,
    now=DEMO_NOW,
    user_mode='question_first',
    question_limit=5,
)

skip_state = copy.deepcopy(RANDOMIZED_STATE)
skip_event = apply_review_action(
    skip_state,
    action='skip_temporarily',
    target_type='wrong_question',
    target_id='q001',
    now=DEMO_NOW,
    skip_days=2.0,
)
skip_plan = build_review_schedule(
    skip_event.updated_payload,
    now=DEMO_NOW,
    user_mode='question_first',
    question_limit=5,
)

print('mark_important 后，错题模式 Top-3：')
for item in important_plan.question_priorities[:3]:
    print('-', item['question_id'], '| score =', item['priority_score'])
print()
print('skip_temporarily 后，错题模式 Top-5：')
for item in skip_plan.question_priorities[:5]:
    print('-', item['question_id'], '| skipped =', item['is_temporarily_skipped'])


mark_important 后，错题模式 Top-3：
- q010 | score = 1.0288
- q005 | score = 0.8979
- q001 | score = 0.8554

skip_temporarily 后，错题模式 Top-5：
- q005 | skipped = False
- q003 | skipped = False
- q010 | skipped = False
- q006 | skipped = False
- q004 | skipped = False
